In [ ]:
# ===== PROJECT TOKEN - do this before running anything =====
# Click  ...  (More menu, top-right)  ->  Insert project token.
# It adds its OWN new Code cell (you type nothing into it).
# If that inserted cell shows as a big heading, click it, press Esc then Y to make it Code, then run it.
# NOTE: this is ONLY the project token (used to read the data).
#       Your API_KEY and SPACE_ID go in Section 2 below - NOT here.

# Demand Forecasting (LightGBM) - Interac transaction-volume forecasting

**How to run - do these in order:**
1. **Insert your project token** at the very top: top-right **... (More) -> Insert project token**, make sure it is a **Code** cell, and run it. (It only reads data - you don't type anything into it.)
2. **Add the data:** open the **"Find and add data"** panel (top-right icon) and **drag-and-drop `training_data_v2.csv`** into the project.
3. **Fill in** `API_KEY`, `SPACE_ID` and `URL` in the **Credentials cell (Section 2)** - that is where your keys go, not the token cell.
4. Choose **Run > Run All Cells**. Do not run cells out of order - the scoring cell at the end needs the earlier cells to run first.

You only ever fill in three values: **URL** (region), **API_KEY**, **SPACE_ID**. Everything else runs as-is.

In [ ]:
# Optional check - confirms the project token was inserted and run. Does NOT crash.
try:
    wslib
    print("OK - project token is set. Continue with Run > Run All Cells.")
except NameError:
    print("No project token yet. Insert it in the cell at the very top (see the README),")
    print("run that cell, then re-run this one. It should then print 'OK - project token is set'.")

## 1. Install packages (run once)
If it says *restart the kernel*, restart, then re-run the project-token cell and continue from the top.

In [ ]:
%pip install -q pandas matplotlib lightgbm "scikit-learn==1.6.*" scipy pyarrow

In [ ]:
import io
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor, early_stopping

## 2. Credentials + deployment space
Set these to match YOUR environment:
- **URL**: your region. Toronto -> `https://ca-tor.ml.cloud.ibm.com`; Dallas -> `https://us-south.ml.cloud.ibm.com`. This must match the account/region shown in the top bar.
- **API_KEY**: an IBM Cloud API key (Manage -> Access (IAM) -> API keys -> Create).
- **SPACE_ID**: a deployment space GUID in the SAME account + region (Deployment spaces -> your space -> Manage -> General).

In [ ]:
from ibm_watsonx_ai import APIClient, Credentials

# Match your region. Dallas is the default below; for Toronto use the ca-tor URL.
URL      = "https://us-south.ml.cloud.ibm.com"   # Dallas.  Toronto: https://ca-tor.ml.cloud.ibm.com
API_KEY  = "PASTE_YOUR_API_KEY"
SPACE_ID = "PASTE_YOUR_SPACE_ID"

if ("PASTE_YOUR" in API_KEY) or ("PASTE_YOUR" in SPACE_ID):
    print("Fill in API_KEY and SPACE_ID above (and set URL to your region), then run this cell again.")
else:
    client = APIClient(Credentials(url=URL, api_key=API_KEY))
    client.set.default_space(SPACE_ID)
    print("Connected. Default space:", SPACE_ID)

## 3. Load training data
This reads the `training_data_v2.csv` data asset from the **project** via `wslib`.
**Add the file first:** top-right **"Find and add data"** panel -> **drag-and-drop `training_data_v2.csv`**.
If you skip this you'll get `No asset named "training_data_v2.csv"`.

In [ ]:
raw = wslib.load_data("training_data_v2.csv")
df_1 = pd.read_csv(raw if hasattr(raw, "read") else io.BytesIO(raw))
print("Loaded:", df_1.shape)
df_1.head()

## 4. Clean + prepare

In [ ]:
if len(df_1) > 50000:
    df_1 = df_1.iloc[50000:]

# This dataset has no date column - every column is a numeric feature or the target.
df = df_1.apply(pd.to_numeric, errors="coerce").fillna(0.0)
if "SEGMENT_ID" in df.columns:
    df = df.sort_values("SEGMENT_ID")
df = df.reset_index(drop=True)
print(df.shape)
df.head()

## 5. Train / validation / test split

In [ ]:
X = df.drop(columns=["target"])
y = df["target"]

# Robust split: last 20% for validation (always non-empty), tail for test.
n = len(df)
n_val = max(1, int(n * 0.2))
X_train, X_val = X.iloc[:-n_val], X.iloc[-n_val:]
y_train, y_val = y.iloc[:-n_val], y.iloc[-n_val:]
X_test, y_test = X.tail(min(1000, n)), y.tail(min(1000, n))
print("train / val / test:", X_train.shape, X_val.shape, X_test.shape)

## 6. (Optional) save train/test back to the project
Uses `wslib.save_data` (the modern replacement for `project.save_data`). Safe to skip.

In [ ]:
train_df = pd.concat([X_train, y_train.rename("target")], axis=1)
test_df  = pd.concat([X_test,  y_test.rename("target")], axis=1)
try:
    wslib.save_data("training_data.csv", train_df.to_csv(index=False).encode(), overwrite=True)
    wslib.save_data("test_data.csv",     test_df.to_csv(index=False).encode(),  overwrite=True)
    print("Saved training_data.csv and test_data.csv to the project.")
except Exception as e:
    print("Skipped saving to project:", e)

## 7. Train the LightGBM model

In [ ]:
def train_lgbm_model(x_train, y_train, x_valid, y_valid):
    model = LGBMRegressor(
        objective="regression",
        n_estimators=1000,
        learning_rate=0.05,
        random_state=42,
        metric="rmse",
    )
    model.fit(
        x_train, y_train,
        eval_set=[(x_valid, y_valid)],
        eval_metric="rmse",
        callbacks=[early_stopping(50)],
    )
    return model

m_lgb = train_lgbm_model(X_train, y_train, X_val, y_val)

## 8. Evaluation report
**Note the RMSE** - you enter it as the OpenScale **Quality** RMSE threshold in the lab's Part B.

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import pearsonr, spearmanr

y_val_pred = m_lgb.predict(X_val)
yt = np.asarray(y_val, dtype=float)
yp = np.asarray(y_val_pred, dtype=float)

rmse = mean_squared_error(yt, yp) ** 0.5
mae  = mean_absolute_error(yt, yp)
r2   = r2_score(yt, yp)
pear = pearsonr(yt, yp)[0]
spear = spearmanr(yt, yp)[0]

nz = yt != 0
mape = float(np.mean(np.abs((yt[nz] - yp[nz]) / yt[nz])) * 100) if nz.any() else float("nan")

print("RMSE     : {:,.2f}   <-- use this as the OpenScale Quality RMSE threshold".format(rmse))
print("MAE      : {:,.2f}".format(mae))
print("R2       : {:.3f}".format(r2))
print("Pearson  : {:.3f}".format(pear))
print("Spearman : {:.3f}".format(spear))
print("MAPE(%)  : {:.2f}".format(mape))

## 9. Register + deploy the model

In [ ]:
software_spec_id = client.software_specifications.get_id_by_name("runtime-25.1-py3.12")

model_meta = {
    client.repository.ModelMetaNames.NAME: "demand_forecasting_lgbm",
    client.repository.ModelMetaNames.TYPE: "scikit-learn_1.6",
    client.repository.ModelMetaNames.SOFTWARE_SPEC_ID: software_spec_id,
}
model_details = client.repository.store_model(model=m_lgb, meta_props=model_meta)
model_id = client.repository.get_model_id(model_details)
print("Model ID:", model_id)

dep_meta = {
    client.deployments.ConfigurationMetaNames.NAME: "demand-forecasting-ml",
    client.deployments.ConfigurationMetaNames.ONLINE: {},
}
dep_details = client.deployments.create(model_id, meta_props=dep_meta)
deployment_id = client.deployments.get_id(dep_details)
print("Deployment SUCCESSFUL - Deployment ID:", deployment_id)

## 10. (Optional) score the deployed model
Uses the SDK and a real row from the test set, so the columns always match. Needs the cells above to have run (Run > Run All Cells).

In [ ]:
if "X_test" not in globals() or "deployment_id" not in globals():
    print("Run the cells above first (Run > Run All Cells) - this cell needs X_test and deployment_id.")
else:
    sample = X_test.iloc[[0]]
    payload = {"input_data": [{"fields": sample.columns.tolist(), "values": sample.values.tolist()}]}
    print(client.deployments.score(deployment_id, payload))